In [11]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import sqlite3

class State(TypedDict):
    text: str

def node_a(state: State): 
    human_resposne = interrupt("Where do you want to go? type b or c ")
    print("human review value:", human_resposne)

    nextnode = None
    if human_resposne.lower() == 'b':
        nextnode = "node_b"
    else:   
        nextnode = "node_c"
    
    return Command(
        goto=nextnode, 
        update={
            "text": state["text"] + "a"
        }
    )

def node_b(state: State): 
    return Command(
        goto="node_c", 
        update={
            "text": state["text"] + "b"
        }
    )

def node_c(state: State): 
    return Command(
        goto=END, 
        update={
            "text": state["text"] + "c"
        }
    )

memory = MemorySaver()
config = {
    "configurable": {
        "thread_id" : 1
    }
}

graph = StateGraph(State)

graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)
graph.add_node("node_c", node_c)

graph.set_entry_point("node_a")

app = graph.compile(checkpointer= memory)

initial_state = {
    "text": ""
}

final_result = app.invoke(initial_state, config, stream_mode="updates")
print(final_result)

[{'__interrupt__': (Interrupt(value='Where do you want to go? type b or c ', id='24aa01faed601c2c5ed8d201023d4d9e'),)}]


In [3]:
app.get_state(config)

StateSnapshot(values={'text': ''}, next=('node_a',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16cece-eb0a-6932-8000-e44a8518459b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-20T21:13:49.783275+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16cece-eb0a-6931-bfff-b0ff37067771'}}, tasks=(PregelTask(id='78c340dd-c2a5-c0b4-1b65-d095bca5e9e9', name='node_a', path=('__pregel_pull', 'node_a'), error=None, interrupts=(Interrupt(value='Where do you want to go? type b or c ', id='7ed5c351827e1d41b976804d4006603b'),), state=None, result=None),), interrupts=(Interrupt(value='Where do you want to go? type b or c ', id='7ed5c351827e1d41b976804d4006603b'),))

In [12]:
user_input = input("Where do you want to go? type b or c")

second_result = app.invoke(
    Command(resume=user_input),
    config=config,
    stream_mode='updates'
)
second_result

human review value: c


[{'node_a': {'text': 'a'}}, {'node_c': {'text': 'ac'}}]